
# Attenuation law leaves a distinct UV-slope fingerprint

For a fixed star-forming galaxy with τ_V = 1 (a moderate
attenuation), six common attenuation laws produce six visibly
different reddened UV slopes β. The intrinsic SED has β ≈ −2.3;
SMC steepens β to ≈ +0.4; Calzetti / Salim leave a flatter
β ≈ −0.5. The spread (~1 mag of UV slope at fixed τ_V) is the
systematic an SED fitter inherits if its dust-law assumption is
wrong.

Pair with ``plot_uv_slope_age`` (intrinsic β vs age) — the two
together explain why the IRX–β diagram is so band-, age-, and
law-dependent in the literature.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_AA_PER_S = 2.998e18
WINDOWS = np.array(
    [
        [1268, 1284],
        [1309, 1316],
        [1342, 1371],
        [1407, 1515],
        [1562, 1583],
        [1677, 1740],
        [1760, 1833],
        [1866, 1890],
        [1930, 1950],
        [2400, 2580],
    ]
)


def _beta_uv(wave, l_nu):
    f_lam = l_nu * C_AA_PER_S / wave**2
    mask = np.zeros_like(wave, dtype=bool)
    for lo, hi in WINDOWS:
        mask |= (wave >= lo) & (wave <= hi)
    slope, _ = np.polyfit(np.log10(wave[mask]), np.log10(f_lam[mask]), 1)
    return float(slope)


LAWS = [
    ("calzetti", "Calzetti+2000", "C0"),
    ("salim", "Salim+2018", "C1"),
    ("cardelli", "Cardelli+1989 MW", "C2"),
    ("smc", "SMC (Pei 1992)", "C3"),
    ("kriek_conroy", "Kriek & Conroy 2013", "C4"),
    ("noll09", "Noll+2009", "C5"),
]
SFH = {
    "type": "tsnorm",
    "all_params": tengri.FIXED,
    "peak_lbt_gyr": 0.05,
    "width_gyr": 0.05,
    "log_total_mass": 10.0,
    "skew": 0.0,
    "trunc": 13.0,
}
ssp = tengri.load_ssp()


def _model(law=None, tau=0.0):
    return tengri.SEDModel.build(
        ssp,
        sfh=SFH,
        dust={
            "type": "two_component",
            "all_params": tengri.FIXED,
            "tau_diff": tau,
            "tau_bc": 0.0,
            "law_diff": law or "calzetti",
        },
        redshift=tengri.Fixed(0.05),
    )


# Intrinsic slope reference
m0 = _model(tau=0.0)
p0 = dict(m0.spec.sample(jax.random.PRNGKey(0)))
out0 = m0.predict(p0)
beta_intrinsic = _beta_uv(np.asarray(m0.wavelengths), np.asarray(out0.rest_sed()))

fig, ax = plt.subplots(figsize=(7.0, 4.6))
ax.axhline(
    beta_intrinsic,
    color="0.55",
    lw=0.7,
    ls="--",
    label=rf"intrinsic ($\beta = {beta_intrinsic:+.2f}$)",
)

for law_key, label, color in LAWS:
    m = _model(law=law_key, tau=1.0)
    p = dict(m.spec.sample(jax.random.PRNGKey(0)))
    out = m.predict(p)
    beta = _beta_uv(np.asarray(m.wavelengths), np.asarray(out.rest_sed()))
    ax.bar(label, beta - beta_intrinsic, color=color, edgecolor="0.15", lw=0.5)
    ax.text(
        label, beta - beta_intrinsic + 0.05, f"{beta:+.2f}", ha="center", fontsize=8, color="0.2"
    )

ax.set(
    ylabel=r"$\Delta\beta$ vs intrinsic  (Calzetti+1994 fit)", ylim=(0, ax.get_ylim()[1] * 1.15)
)
ax.set_xticklabels([lbl for _, lbl, _ in LAWS], rotation=20, ha="right", fontsize=8.5)
ax.legend(frameon=False, fontsize=9, loc="upper right")
ax.text(
    0.02,
    0.97,
    r"applied $\tau_V = 1$",
    transform=ax.transAxes,
    fontsize=8,
    color="0.4",
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.8", lw=0.4),
    va="top",
)

fig.tight_layout()
plt.savefig("plot_dust_law_uv_slope_response.png", dpi=150, bbox_inches="tight")